# Proyecto de Estructuras de Datos

Para esta libreta, se tiene contemplado que la función de recolección de datos se encuentra en archivos .py en la carpeta src/ Estos archivos se ejecutan en terminal la libreta llama a los archivos resultantes almacenados en data/

## Configuración Inicial
Se realizan configuraciones iniciales para comenzar a cargar datos desde la API de Semantic Scholar.
Si se obtiene una API KEY válida, puede comenzarse la extracción de datos y almacenamiento de archivos en formato .csv

In [17]:
import os
from dotenv import load_dotenv

load_dotenv()

SEMANTIC_SCHOLAR_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY")
API_URL = "https://api.semanticscholar.org/graph/v1/paper/search"

if SEMANTIC_SCHOLAR_API_KEY:
    HEADERS = {"x-api-key": SEMANTIC_SCHOLAR_API_KEY}
    USE_API = True
    print("API key encontrada. Se podrá recolectar desde Semantic Scholar.")
else:
    HEADERS = {}
    USE_API = False
    print("No se encontró API key. Se usarán los CSV locales ya generados.")

API key encontrada. Se podrá recolectar desde Semantic Scholar.


## Recolección de Datos
Se inicial las operaciones para poder obtener los papers necesarios para el vaciado de autores y relaciones de trabajo
Primero se cargan librerías y directorios donde estarán almacenados los archivos tras la recolección

In [20]:
import sys
import os
sys.path.insert(0, os.getcwd())
import time
import requests
import pandas as pd

#Archivos y constantes para el proyecto
DATA_DIR = os.path.join(os.getcwd(), "data")
RAW_PAPER_FILE = os.path.join(DATA_DIR, "raw_papers.csv")
PROCESSED_PAPER_FILE = os.path.join(DATA_DIR, "processed_papers.csv")
AUTHORS_FILE = os.path.join(DATA_DIR, "authors.csv")

#Palabras clave para búsqueda de artículos dentro de la API
QUERIES = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "neural networks"
]

Se definen las funciones principales para la recolección de datos

In [28]:
def create_data_dir():
    os.makedirs(DATA_DIR, exist_ok=True)

#Consulta de artículos
def fetch_papers(query, limit=25):
    params = {
        "query": query,
        "limit": limit,
        "fields": "paperId,title,year,authors"
    }
    try: 
        response = requests.get(API_URL, params=params, headers=HEADERS, timeout=30)
        print(f"Consulta: {query} - Estatus: {response.status_code}")
        print (f"Respuesta: {response.status_code}")  
        #El número de respuesta 200 es una consulta exitosa
        if response.status_code ==200:
            return response.json().get("data", [])
        #El número de respuesta 429 notifica que se ha llegado al límite de requests
        if response.status_code == 429:
            print("Límite de tasa alcanzado. Esperando 60 segundos...")
            time.sleep(60)
            return fetch_papers(query, limit)
    except requests.exceptions.RequestException as e:
        print(f"Error al consultar la API: {e}")
    return []

#Formato de artículos guardados sin aplicar ningún filtro
def save_raw_data():
    all_papers = []
    for query in QUERIES:
        papers = fetch_papers(query,limit=25)
        for paper in papers:
            paper["query_used"] = query
            all_papers.append(paper)
        print(f"Artículos obtenidos: {len(papers)}")
        print("="*40)
        time.sleep(1)  # Para evitar exceder el límite de tasa
    
    return all_papers

#Aplicación de filtros para almacenar únicamente la información relevante:
#- Identificación del artículo (paperId)
#- Título del artículo (title)
#- Año de publicación (year)
#- Lista de autores (authors)

def clean_data(papers):
    cleaned = []
    for paper in papers:
        paper_id = paper.get("paperId")
        title = paper.get("title")
        year = paper.get("year")
        authors = paper.get("authors", [])
        
        if not paper_id or not title or not year:
            print(f"Artículo ignorado (datos faltantes): {paper}")
            continue
        if not isinstance(authors, list) or len(authors) < 2:
            print(f"Artículo ignorado (autores insuficientes): {title}")
            continue
        cleaned.append({
            "paperId": paper_id,
            "title": title,
            "year": year,
            "authors": authors
        })   
    df_clean = pd.DataFrame(cleaned)
    if not df_clean.empty:
         df_clean = df_clean.drop_duplicates(subset=["paperId"])
    return df_clean


#De los artículos obtenidos, se filtran los autores con sus posteriores campos de identificación
def extract_authors(papers):
    authors = []
    for _, paper in papers.iterrows():
        for author in paper["authors"]:
            author_id = author.get("authorId")
            name = author.get("name")
            if author_id and name:
                authors.append({"authorId": author_id, "name": name})
    df_authors = pd.DataFrame(authors)
    if not df_authors.empty:
        df_authors = df_authors.drop_duplicates(subset=["authorId"])
    # Limitar el número de autores guardados para evitar conjuntos demasiado grandes
    if not df_authors.empty:
        df_authors = df_authors
    return df_authors
 
#Función principal que ejecuta todo el proceso de recolección, limpieza y extracción de datos, además de guardar los resultados en archivos CSV para su posterior análisis. Se incluyen mensajes de impresión para monitorear el progreso y la cantidad de datos obtenidos en cada etapa.
def run_collection():
    create_data_dir()
    raw_papers = save_raw_data()
    df_raw = pd.DataFrame(raw_papers)
    df_raw.to_csv(RAW_PAPER_FILE, index=False, encoding="utf-8-sig")
    df_cleaned = clean_data (raw_papers)
    df_cleaned.to_csv(PROCESSED_PAPER_FILE, index=False, encoding="utf-8-sig")
    print (f"Articulos limpios obtenidos: {len(df_cleaned)}")
    df_authors = extract_authors(df_cleaned)
    df_authors.to_csv(AUTHORS_FILE, index=False, encoding="utf-8-sig")
    print (f"Autores extraidos: {len(df_authors)}")
    print("Recoleccion terminada")

Se llama a la función y comienza la recolección de datos desde la API

In [29]:
run_collection()

Consulta: artificial intelligence - Estatus: 200
Respuesta: 200
Artículos obtenidos: 25
Consulta: machine learning - Estatus: 200
Respuesta: 200
Artículos obtenidos: 25
Consulta: deep learning - Estatus: 200
Respuesta: 200
Artículos obtenidos: 25
Consulta: neural networks - Estatus: 200
Respuesta: 200
Artículos obtenidos: 25
Artículo ignorado (autores insuficientes): Explanation in Artificial Intelligence: Insights from the Social Sciences
Artículo ignorado (autores insuficientes): High-performance medicine: the convergence of human and artificial intelligence
Artículo ignorado (autores insuficientes): Adaptation in Natural and Artificial Systems: An Introductory Analysis with Applications to Biology, Control, and Artificial Intelligence
Artículo ignorado (autores insuficientes): Is it artificial intelligence or real?
Artículo ignorado (autores insuficientes): Machine Learning: Algorithms, Real-World Applications and Research Directions
Artículo ignorado (autores insuficientes): Stop E

## Construcción de edges 
Se realizan imports y se definen constantes para establecer directorios

In [38]:
import os
import ast
import pandas as pd
from itertools import combinations
from collections import defaultdict


DATA_DIR = os.path.join(os.getcwd(), "data")
CLEAN_PAPERS_FILE = os.path.join(DATA_DIR, "processed_papers.csv")
AUTHORS_FILE = os.path.join(DATA_DIR, "authors.csv")
EDGES_FILE = os.path.join(DATA_DIR, "edges.csv")


Funcion para parsear autores

In [39]:
def parse_authors(authors_text):
    try:
        #authors crea un diccionario con ast, que convierte la cadena en una estructura de datos de Python
        authors = ast.literal_eval(authors_text)
        #ifinstance verifica si authors es una lista, lo que es esperado. Si es así, se devuelve la lista de autores. 
        #caso contrario, devuelve una lista vacía
        if isinstance(authors, list):
            return authors
    except Exception:
        return []

    return []

In [51]:
def build_edges():
    df_papers = pd.read_csv(CLEAN_PAPERS_FILE)
    df_authors = pd.read_csv(AUTHORS_FILE)

    # Autores permitidos: son los autores oficiales del grafo.
    allowed_authors = set(df_authors["authorId"].astype(str))

    edge_weights = defaultdict(int)
    edge_names = {}
    #Se busca en cada artículo recolectado. Iterrows permite iterar sobre cada fila del DataFrame
    for _, paper in df_papers.iterrows():
        authors = parse_authors(paper["authors"])

        clean_authors = []

        for author in authors:
            author_id = author.get("authorId")
            name = author.get("name")

            if author_id and name:
                author_id = str(author_id)

                # Solo se conservan autores que estén dentro del subconjunto oficial.
                if author_id in allowed_authors:
                    clean_authors.append((author_id, name))

        # Si dentro del artículo hay menos de 2 autores del subconjunto oficial,
        # no se puede crear una arista.
        if len(clean_authors) < 2:
            continue

        # Crear colaboraciones reales entre pares de autores del mismo artículo.
        for author1, author2 in combinations(clean_authors, 2):
            id1, name1 = author1
            id2, name2 = author2
            #Sorted ordena los identificadores de los autores para garantizar que la arista sea consistente 
            source, target = sorted([id1, id2])

            if source == id1:
                source_name = name1
                target_name = name2
            else:
                source_name = name2
                target_name = name1
            #cada vez que se encuentra una colaboración entre dos autores, se incrementa el peso de la arista y se registra el nombre
            edge = (source, target)

            edge_weights[edge] += 1
            edge_names[edge] = (source_name, target_name)

    edges = []
    #Este for itera sobre cada arista encontrada, obteniendo el peso y los nombres de los autores para luego crear un diccionario 
    for (source, target), weight in edge_weights.items():
        source_name, target_name = edge_names[(source, target)]

        edges.append({
            "source": source,
            "target": target,
            "weight": weight,
            #Se calcula la distancia como el inverso del peso, lo que significa que una mayor
            #  colaboración (peso) se traduce en una menor distancia entre los autores en el grafo.
            "distance": 1 / weight,
            "source_name": source_name,
            "target_name": target_name
        })

    df_edges = pd.DataFrame(edges)
    df_edges.to_csv(EDGES_FILE, index=False, encoding="utf-8-sig")

    print("Archivo de aristas creado.")
    print(f"Autores oficiales del grafo: {len(allowed_authors)}")
    print(f"Total de aristas creadas: {len(df_edges)}")
    return df_edges



In [54]:
build_edges()

Archivo de aristas creado.
Autores oficiales del grafo: 621
Total de aristas creadas: 4590


,source,target,weight,distance,source_name,target_name
0,1379511816,2058921025,1,1.0,Alejandro Barredo Arrieta,Natalia Díaz Rodríguez
1,1379511816,9221552,1,1.0,Alejandro Barredo Arrieta,J. Ser
2,1379511786,1379511816,1,1.0,Adrien Bennetot,Alejandro Barredo Arrieta
3,1379511816,3030006,1,1.0,Alejandro Barredo Arrieta,S. Tabik
4,1379511816,50449165,1,1.0,Alejandro Barredo Arrieta,A. Barbado
...,...,...,...,...,...,...
4585,143787583,2004053,1,1.0,Ali Farhadi,Vicente Ordonez
4586,143787583,40497777,1,1.0,Ali Farhadi,J. Redmon
4587,1746183,1891828,1,1.0,Alexander S. Ecker,Leon A. Gatys
4588,1731199,1891828,1,1.0,M. Bethge,Leon A. Gatys


## Construcción del grafo
A partir de los edges creados con los archivos obtenidos, se crea el grafo principal y se almacena para ser analizado por Networkx

In [ ]:
import os
import pickle
import pandas as pd
import networkx as nx

DATA_DIR = "data"
OUTPUT_DIR = "output"
AUTHORS_FILE = os.path.join(DATA_DIR, "authors.csv")
EDGES_FILE = os.path.join(DATA_DIR, "edges.csv")
#extensión .gpickle es un formato específico de NetworkX para guardar grafos de manera eficiente
GRAPH_FILE = os.path.join(OUTPUT_DIR, "research_graph.gpickle")

#Función para crear el directorio de salida si no existe
def create_output_dir():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

Se definen las funciones principales 

In [42]:
#Función para construir el grafo a partir de los archivos de autores y aristas. Se utiliza la biblioteca NetworkX para crear un grafo no dirigido
def build_graph():
    df_authors = pd.read_csv(AUTHORS_FILE)
    df_edges = pd.read_csv(EDGES_FILE)
    G = nx.Graph()
    # Agregar nodos de autores al grafo
    for _, row in df_authors.iterrows():
        author_id = row["authorId"]
        name = row["name"]
        G.add_node(str(row["authorId"]), name=row["name"]) 
    # Agregar aristas de colaboración al grafo
    for _, row in df_edges.iterrows():
        source = str(row["source"])
        target = str(row["target"])
        weight = int(row["weight"])
        source_name = row["source_name"]
        target_name = row["target_name"]
        G.add_edge(source, target, weight=weight, source_name=source_name,
                    target_name=target_name)
    return G
    
def save_graph(graph):
    create_output_dir()
    with open(GRAPH_FILE, "wb") as f:
        #picke.dump serializa un objeto y lo guarda en un archivo
        #usa la extensión .gpickle que networkx utiliza para guardar grafos 
        pickle.dump(graph, f)

In [43]:
#Ejecución de la función principal para construir el grafo y guardarlo en un archivo
def main():
    graph = build_graph()
    save_graph(graph)
    print(f"Grafo guardado en: {GRAPH_FILE}")
main()

Grafo guardado en: output\research_graph.gpickle


## Análisis del grafo 
A partir del grafo construido, se efectúan algoritmos para analizar la conectividad del grafo, las comunidades identificadas, caminos cortos entre autores, centralidad de nodos y nodos aislados.
Primero se cargan las librerías utilizadas, directorios y funciones auxiliares

In [45]:
import os
import pickle
from matplotlib import cm
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

#Importar la función greedy_modularity_communities para detectar comunidades dentro del grafo utilizando el algoritmo de modularidad codiciosa
from networkx.algorithms.community import greedy_modularity_communities

OUTPUT_DIR = "output"
FIGURES_DIR = "figures"
GRAPH_FILE = os.path.join(OUTPUT_DIR, "research_graph.gpickle")
CENTRALITY_FILE = os.path.join(OUTPUT_DIR, "centrality.csv")
COMPONENTS_FILE = os.path.join(OUTPUT_DIR, "components.csv")
ISOLATED_AUTHORS_FILE = os.path.join(OUTPUT_DIR, "isolated_authors.csv")
COMMUNITIES_FILE = os.path.join(OUTPUT_DIR, "communities.csv")
FULL_GRAPH_IMAGE = os.path.join(FIGURES_DIR, "full_graph.png")
IMPORTANT_NODES_IMAGE = os.path.join(FIGURES_DIR, "important_nodes.png")
GEXF_FILE = os.path.join(OUTPUT_DIR, "research_graph.gexf")
GEXF_IMPORTANT_NODES_FILE = os.path.join(OUTPUT_DIR, "important_nodes.gexf")

#Función para cargar el grafo desde un archivo utilizando pickle
def load_graph():
    with open(GRAPH_FILE, "rb") as file:
        G = pickle.load(file)
    return G

#Función para obtener el nombre del autor a partir de su identificador en el grafo. 
# Se accede a los atributos del nodo para recuperar el nombre, y si no está disponible, se devuelve "No disponible".
def get_author_name(G,node_id):
    return G.nodes[node_id].get("name", "No disponible")

Analizamos la información general del grafo

In [48]:
def analyze_graph(G):
    print("Informacion general del grafo")
    print(f"Número de nodos: {G.number_of_nodes()}")
    print(f"Número de aristas: {G.number_of_edges()}")
    print(f"Densidad del grafo: {nx.density(G):.4f}")
    isolated_authors = list(nx.isolates(G))
    print(f"Número de autores aislados: {len(isolated_authors)}")
    print(f"Autores aislados: {[get_author_name(G, author) for author in isolated_authors]}")
    components = list(nx.connected_components(G))
    print(f"Número de componentes conectados: {len(components)}")
    print(f"Tamaño de la componente más grande: {len(max(components, key=len))}")
    print(f"Autores en la componente más grande: {[get_author_name(G, author) for author in max(components, key=len)]}\n")

analyze_graph(load_graph())

Informacion general del grafo
Número de nodos: 621
Número de aristas: 4590
Densidad del grafo: 0.0238
Número de autores aislados: 0
Autores aislados: []
Número de componentes conectados: 62
Tamaño de la componente más grande: 249
Autores en la componente más grande: ['Tianfan Fu', 'Bharath Ramsundar', 'Neil C. Rabinowitz', 'Victoria Langston', 'K. Kavukcuoglu', 'Juraj Gottweis', 'Pushmeet Kohli', 'Vihan Jain', 'Sasank Chilamkurthy', 'Xiaoqiang Zhang', 'Jimeng Sun', 'Yann LeCun', 'Weihua Hu', 'Kunal Talwar', 'T. Graepel', 'H. Blau', 'Lukasz Kaiser', 'Lichan Hong', 'M. Wicke', 'S. Jha', 'James Bradbury', 'Linfeng Zhang', 'D. Kumaran', 'Anima Anandkumar', 'Zahra F H Abad', 'Oishi Banerjee', 'William L. Hamilton', 'Ashish Agarwal', 'T. Shaked', 'Yoshua Bengio', 'A. Esteva', 'Anil Palepu', 'Rohan Anil', 'S. Jegelka', 'Quoc V. Le', 'O. Vinyals', 'Alexandre Robicquet', 'C. Citro', 'Kaifeng Chen', 'M. Kudlur', 'A. Swami', 'A. Karthikesalingam', 'George van den Driessche', 'Le Song', 'G. Anders

Ahora, analizamos los componentes, es decir, un subconjunto de nodos que están conectados entre sí

In [ ]:
def analyze_components(G):
    components = list(nx.connected_components(G))
    results = []
    for i, component in enumerate(components, start=1):
        for node in component:
            results.append({
                "authorId": node,
                "name": get_author_name(G, node),
                "component": i,
                "component_size": len(component)
            })
    df_components = pd.DataFrame(results)
    df_components.to_csv(COMPONENTS_FILE, index=False, encoding="utf-8-sig")
    #funcion isolates de nx para identificar nodos sin conexiones
    isolated_nodes = list(nx.isolates(G))
    isolated_rows = []

    for node in isolated_nodes:
        isolated_rows.append({
            "authorId": node,
            "name": get_author_name(G, node)
        })
    df_isolated = pd.DataFrame(isolated_rows)
    df_isolated.to_csv(ISOLATED_AUTHORS_FILE, index=False, encoding="utf-8-sig")
    component_sizes = sorted ([len(c) for c in components], reverse=True)
    print(f"Tamaño de las componentes conectadas: {component_sizes}")
